# The Creative Prisim
## Phase 0 — Engine Verification
**The Intelligence of Seeing (IOS)**

This notebook verifies that the Creative Prisim engine is correctly installed
and operational. It imports from `engine.py` and runs a single smoke test.

**Run this once before starting Phase 1.**

---
### What this confirms:
- `engine.py` imports cleanly
- API key is loaded from `.env`
- All prompt files are present in `prompts/`
- One live Creator API call succeeds
- Blackboard records the session correctly
- Session saves to `sessions/` as JSON

---
## Cell 1 — Install Dependencies
Run once. Restart kernel if prompted.

In [ ]:
%pip install anthropic python-dotenv --quiet

---
## Cell 2 — Import Engine + Verify

Imports everything from `engine.py` and runs the verification check.
You should see green checkmarks for all prompt files and the API client.

In [ ]:
from engine import (
    client,
    create_blackboard,
    scribe_log,
    build_prompt,
    call_role,
    save_session,
    verify_engine,
    ROLE_WEIGHTS,
    GLOBAL_MODES,
    apply_global_mode,
    DEFAULT_MODEL
)

import json

# Run engine verification — checks all files and API client
engine_ready = verify_engine()

if not engine_ready:
    raise RuntimeError("Engine verification failed. Fix missing files before proceeding.")

---
## Cell 3 — Smoke Test: Live Creator Call

One real API call through the Creator role to confirm the full stack works:
engine → prompt compiler → API → Blackboard → session save.

**Prompt:** *Design a brand identity for a new independent bookstore café.*

In [ ]:
# Initialize session
session_prompt = "Design a brand identity for a new independent bookstore café."
blackboard = create_blackboard(session_prompt)
scribe_log(blackboard, "SYSTEM", "session_start",
           f"Smoke test session: '{session_prompt}'")

print(f"Session ID : {blackboard['session_metadata']['session_id']}")
print(f"Prompt     : {session_prompt}")
print(f"Model      : {DEFAULT_MODEL}")
print("Calling API...\n")

# Creator smoke test
creator_response = call_role(
    role="creator",
    user_message=(
        "Generate 3 distinct conceptual directions for this brand identity. "
        "For each: a short title, 2-sentence concept, and emotional territory. "
        "Include one unexpected direction."
    ),
    weights=ROLE_WEIGHTS["creator"],
    context=session_prompt,
    blackboard=blackboard
)

blackboard["ideation_cycles"].append({
    "cycle_number": 1,
    "creator_proposals": [{"raw_response": creator_response}],
    "critic_feedback": [],
    "synthesized_directions": []
})

print("── CREATOR RESPONSE ────────────────────────────────────────")
print(creator_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ API call successful")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 4 — Save and Confirm

In [ ]:
saved_path = save_session(blackboard)

print(f"✓ Session saved: {saved_path}")
print()
print("── REASONING TRACE ─────────────────────────────────────────")
for entry in blackboard["reasoning_trace"]:
    print(f"  {entry['step']:>2} | {entry['role']:<12} | {entry['action']:<18} | {entry['summary'][:60]}")
print()
print("✓ Phase 0 verification complete. Engine is operational.")
print("  Open creative_prisim_phase1.ipynb to run a full studio session.")